# SynBioCrow 2.3 — Galaxy Held-Out Benchmark: Frozen-Policy Scoring\nRun only after the held-out prediction package is sealed. Verifies the prediction hash and frozen 2D-primary similarity-policy hash before accessing the 65-path benchmark truth. The benchmark split is never used for learning or policy revision during scoring.\n

In [ ]:
import sys,subprocess,shutil,json,zipfile,hashlib,time
from pathlib import Path
from google.colab import drive, files

drive.mount("/content/drive",force_remount=False)

REPO=Path("/content/SynBioCrow_GALAXY_BENCH_SCORE")
NORM=Path("/content/drive/MyDrive/SynBioCrow/benchmark/2.3/galaxy_literature_normalized/galaxy_literature_benchmark_normalized.json")
SEALED=Path("/content/drive/MyDrive/SynBioCrow/benchmark/2.3/galaxy_benchmark_sealed")
PANEL=SEALED/"galaxy_benchmark_truth_stripped_panel.json"
MANIFEST=SEALED/"sealed_manifest.json"
OUT=Path("/content/drive/MyDrive/SynBioCrow/benchmark/2.3/galaxy_benchmark_scored")
OUT.mkdir(parents=True,exist_ok=True)

def stream(cmd,cwd=None,timeout=7200):
    print("$"," ".join(map(str,cmd)),flush=True)
    p=subprocess.Popen(list(map(str,cmd)),cwd=cwd,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,bufsize=1)
    t0=time.time(); lines=[]
    for line in p.stdout:
        print(line.rstrip(),flush=True); lines.append(line)
        if time.time()-t0>timeout:
            p.kill(); raise subprocess.TimeoutExpired(cmd,timeout)
    rc=p.wait()
    if rc: raise subprocess.CalledProcessError(rc,cmd,output="".join(lines))

for p in (NORM,SEALED,PANEL,MANIFEST):
    assert p.exists(),p

if REPO.exists(): shutil.rmtree(REPO)
stream(["git","clone","--depth","1","--branch","develop/2.3",
        "https://github.com/theiman112860/SynBioCrow.git",str(REPO)],timeout=600)
stream([sys.executable,"-m","pip","install","-q","-e",str(REPO)],timeout=1200)

SCRIPT=REPO/"scripts/evaluate_galaxy_benchmark_2_3.py"
POLICY=REPO/"docs/SIMILARITY_POLICY_2_3.md"
stream([sys.executable,"-m","py_compile",str(SCRIPT)],timeout=120)

manifest=json.loads(MANIFEST.read_text())
actual=hashlib.sha256((SEALED/"sealed_predictions.json").read_bytes()).hexdigest()
assert actual==manifest["predictions_sha256"],(actual,manifest["predictions_sha256"])

seal=json.loads((SEALED/"benchmark_seal.json").read_text())
policy_sha=hashlib.sha256(POLICY.read_bytes()).hexdigest()
assert policy_sha==seal["similarity_policy_sha256"],(policy_sha,seal["similarity_policy_sha256"])
assert seal["truth_accessed_during_prediction"] is False
print("[PREFLIGHT] prediction seal PASS",actual)
print("[PREFLIGHT] frozen similarity policy PASS",policy_sha)

result=OUT/"galaxy_benchmark_evaluation.json"
stream([sys.executable,str(SCRIPT),
        "--benchmark",str(NORM),
        "--panel",str(PANEL),
        "--predictions-dir",str(SEALED),
        "--manifest",str(MANIFEST),
        "--policy-file",str(POLICY),
        "--output",str(result)],cwd=REPO,timeout=7200)

payload=json.loads(result.read_text())
assert payload["benchmark_denominator"]==65
assert payload["runnable_targets"]==64
assert payload["mapping_limited_targets"]==1
assert payload["benchmark_truth_used_for_learning"] is False

bundle=OUT/"SynBioCrow_2_3_GALAXY_BENCHMARK_SCORED_RESULTS.zip"
with zipfile.ZipFile(bundle,"w",zipfile.ZIP_DEFLATED) as z:
    z.write(result,result.name)
    z.write(SCRIPT,"evaluate_galaxy_benchmark_2_3.py")
    z.write(POLICY,"SIMILARITY_POLICY_2_3.md")
print("BUNDLE",bundle)
print("SHA256",hashlib.sha256(bundle.read_bytes()).hexdigest())
files.download(str(bundle))
